In [3]:
import requests
import pandas as pd
import time

# Load your data
df = pd.read_csv('./all_singapore.csv')

# Function to query OpenAlex API
def get_affiliations_by_doi(doi):
    try:
        url = f"https://api.openalex.org/works/https://doi.org/{doi}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            authorships = data.get('authorships', [])
            
            if not authorships:
                return None, None

            # First author
            first_author_info = authorships[0]
            first_author_affiliations = [inst.get('display_name') for inst in first_author_info.get('institutions', [])]

            # Other authors
            other_authors_info = authorships[1:]
            other_affiliations = []
            for author in other_authors_info:
                for inst in author.get('institutions', []):
                    other_affiliations.append(inst.get('display_name'))

            # Remove duplicates
            first_author_affiliations = list(set(first_author_affiliations))
            other_affiliations = list(set(other_affiliations))

            return first_author_affiliations, other_affiliations
        else:
            print(f"Failed to fetch DOI: {doi}")
            return None, None
    except Exception as e:
        print(f"Error with DOI {doi}: {e}")
        return None, None

# Create two new columns
df['Affiliation (First Author)'] = None
df['Affiliation (Authors)'] = None

# Go through each row and fetch affiliations
for idx, row in df.iterrows():
    doi = row['Item DOI']
    first_affil, other_affils = get_affiliations_by_doi(doi)
    
    df.at[idx, 'Affiliation (First Author)'] = first_affil
    df.at[idx, 'Affiliation (Authors)'] = other_affils

    time.sleep(1)  # Be polite to the API

# Save to a new CSV file
df.to_csv('./all_singapore_with_affiliations.csv', index=False)

print("✅ Done! Affiliations added and file saved.")

✅ Done! Affiliations added and file saved.


In [4]:
df

,Item Title,Publication Title,Item DOI,Authors,Publication Year,First Author,Gender (First Author),Gender likelihood (First Author),Gender (Authors),Gender likelihood (Authors),Affiliation (First Author),Affiliation (Authors)
0,Linker-assisted assembly of 1D TiO2 nanobelts/...,Applied Catalysis A: General,10.1016/j.apcata.2015.10.046,"['Fangxing Xiao', 'Guangcan Xiao', 'Bin Liu']",2016,Junyu Zhang,male,0.610558,"['female', 'female', 'female']","[0.5682396883005217, 0.6465339974345546, 0.540...",[Fuzhou University],"[Fuzhou University, Nanyang Technological Univ..."
1,Zr-doped TiO2 for enhanced photocatalytic degr...,Applied Catalysis A: General,10.1016/j.apcata.2009.12.025,"['Tuti Mariana Lim', 'Dewi Puspitaningrum Suba...",2010,Bifen Gao,female,0.622666,"['female', 'female', 'female']","[0.7546240118988579, 0.6812576004715208, 0.568...",[Nanyang Technological University],[Nanyang Technological University]
2,Effects of K-dopant on structure and activity ...,Applied Catalysis A: General,10.1016/j.apcata.2016.04.015,"['Qinghu Tang', 'Jia Zhang', 'Jing Wang', 'Pei...",2016,Xiaopei Xie,male,0.531764,"['female', 'female', 'female', 'female', 'fema...","[0.5470140976472981, 0.691293533715916, 0.6840...",[Henan Normal University],"[Institute of Process Engineering, Institute o..."
3,Ethene hydrogenation on zeolite-supported rhod...,Applied Catalysis A: General,10.1016/j.apcata.2017.06.036,"['Georgi N. Vayssilov', 'Alexander Genest', 'N...",2017,Velina K. Markova,female,0.972108,"['female', 'female', 'female']","[0.6359252386136343, 0.6506455578518288, 0.536...","[Agency for Science, Technology and Research, ...","[Institute of High Performance Computing, Tech..."
4,Enhanced catalytic performances of Ni/Al2O3 ca...,Applied Catalysis A: General,10.1016/j.apcata.2014.09.028,"['Fangna Gu', 'Xiaopeng Lu', 'Youjun Liu', 'Hu...",2014,Qing Liu,female,0.666123,"['female', 'female', 'female', 'female', 'fema...","[0.6294367108419282, 0.6879814401887265, 0.608...",[Institute of Process Engineering],"[Institute of Process Engineering, Institute o..."
...,...,...,...,...,...,...,...,...,...,...,...,...
915,Investigating coating method induced vertical ...,Solar Energy Materials and Solar Cells,10.1016/j.solmat.2017.12.007,"['Chellappan Vijila', 'Wei Peng Goh', 'Jie Zha...",2018,Changyun Jiang,male,0.539541,"['male', 'female', 'female']","[0.6732412951748872, 0.6626843477126031, 0.640...",[],[Changzhou Institute of Printed Electronics In...
916,Improved photoelectrochemical property of a na...,Solar Energy Materials and Solar Cells,10.1016/j.solmat.2014.08.015,"['Xiaoli Zhang', 'Shaohong Jiang', 'Haitao Dai...",2015,Yang Li,female,0.556293,"['female', 'female', 'female', 'female', 'fema...","[0.5482550118509895, 0.6797641668335582, 0.653...",[Tianjin University],"[Tianjin University, Nanyang Technological Uni..."
917,DSSCs with ZnO@TiO2 core–shell photoanodes sho...,Solar Energy Materials and Solar Cells,10.1016/j.solmat.2016.05.009,"['D. S.U. Peiris', 'Piyasiri Ekanayake', 'Ai L...",2016,Rajour Tanyi Ako,male,0.606005,"['male', 'female', 'female', 'male', 'female',...","[0.9003927192675811, 0.6923669543457026, 0.761...",[Universiti Brunei Darussalam],"[Universiti Brunei Darussalam, University of t..."
918,Morphology and electrical characteristics of p...,Solar Energy Materials and Solar Cells,10.1016/j.solmat.2018.04.013,"['Zui Tao', 'Weiwei Yang', 'Kaichen Xu', 'Lu W...",2018,Xinyan Zhao,female,0.679233,"['male', 'female', 'female', 'female', 'female...","[0.544495678628407, 0.6866559009165518, 0.6545...",[Southern University of Science and Technology...,"[Virginia Tech, Southern University of Science..."
